• Loaded raw datasets and standardised column names, data types, and formats  
• Removed duplicate booking and restaurant records to avoid double-counting  
• Filtered booking data to keep only completed and realised transactions  
• Excluded cancelled, refunded, no-show, inactive, system-generated, and future-dated records  
• Dropped columns with excessive missing values and removed rows with missing critical identifiers  
• Identified and removed extreme outliers according to identified restaurant_id in gsheet  
• Converted revenue values from cents to dollars for consistency  
• Merged booking-level data with restaurant metadata using restaurant_id , used geolocation api to retrieve countries and information about restaurant
• Aggregated booking-level data to restaurant-level performance metrics  
• Created time features (year, month) from booking dates  
• Engineered growth, trend, and rolling metrics for momentum analysis  


In [54]:
import sys
print(sys.executable)

c:\Users\lowth\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [55]:
# If you have yet installed the libraries
# !{sys.executable} -m pip install pyarrow fastparquet


In [56]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = Path.cwd().parent  
PARQUET_PATH = BASE_PATH / "data" / "mv_dataset_parquet"

bookings = pd.read_parquet(PARQUET_PATH / "bookings.parquet")
restaurants = pd.read_parquet(PARQUET_PATH / "restaurants.parquet")
cuisine = pd.read_parquet(PARQUET_PATH / "cuisine.parquet")
ratings = pd.read_parquet(PARQUET_PATH / "ratings.parquet")
channels = pd.read_parquet(PARQUET_PATH / "channels.parquet")


In [57]:
bookings.isnull().sum() 

id                           0
restaurant_id                0
date                         0
start_time                   0
end_time                     0
adult                        0
kids                         0
user_id_masked               0
guest_id_masked        3267512
medium                 2562347
channel                      0
arrived                      0
no_show                      0
for_locking_system           0
is_temporary                 0
active                       0
has_special_request    2582512
created_at                   0
ack                          0
cancelled_by           3167337
adjusted                     0
refund                       1
confirmed_by           3266089
prepared                     0
revenue                      0
has_promptpay          3172193
has_cc                 3109817
has_shopee_pay         2588914
has_alipay             3264530
has_wechat_pay         3265440
has_true_wallet        3267992
dtype: int64

In [58]:
bookings['active'].unique()

array([ True, False])

In [59]:
valid_bookings = bookings.copy()

In [60]:
#This section documents how Hungry Hub filters their data for Gross Merchandise Value (GMV) calculations, e.g. no_show, refund, and revenue, and the list continues.
#I presume there should be more restrictions that we have to set, so please add accordingly. 

#i think with this new dataset, some of the fields have changed data types, we may need to edit the code separately. for now, i will create valid_bookings_df as a copy of bookings
# valid_bookings = bookings[
#     (bookings['revenue'] > 0) &
#     (bookings["no_show"] == False) &
#     (bookings["refund"] == 'False') &
#     (bookings['active'] == True) &
#     (bookings['ack'] == True) &
#     (bookings['is_temporary'] == False) &
#     (bookings['for_locking_system'] == True) &
#     (bookings['channel'] != 5)
# ]
# valid_bookings.head()

In [61]:
valid_bookings.describe()

,id,restaurant_id,adult,kids,guest_id_masked,channel,has_special_request,revenue,has_promptpay,has_cc,has_shopee_pay,has_alipay,has_wechat_pay,has_true_wallet
count,3.267992e+06,3.267992e+06,3.267992e+06,3.267992e+06,4.800000e+02,3.267992e+06,685480.000000,3.267992e+06,95799.0,158175.0,679078.0,3462.0,2552.0,0.0
mean,6.195581e+06,3.113601e+03,2.868470e+00,1.003528e-01,9.049160e+18,1.836904e+02,0.138907,3.757548e+04,1.0,1.0,0.0,1.0,1.0,NaN
std,1.255942e+06,1.610956e+03,2.281271e+00,1.389173e+01,5.198629e+18,1.096734e+03,0.345850,8.319358e+04,0.0,0.0,0.0,0.0,0.0,NaN
min,8.657320e+05,3.300000e+01,0.000000e+00,-1.000000e+00,3.748906e+16,0.000000e+00,0.000000,1.000000e+01,1.0,1.0,0.0,1.0,1.0,NaN
25%,5.159982e+06,1.561000e+03,2.000000e+00,0.000000e+00,4.748193e+18,0.000000e+00,0.000000,1.558800e+04,1.0,1.0,0.0,1.0,1.0,NaN
50%,6.142960e+06,3.618000e+03,2.000000e+00,0.000000e+00,8.829972e+18,0.000000e+00,0.000000,2.900000e+04,1.0,1.0,0.0,1.0,1.0,NaN
75%,7.163924e+06,4.503000e+03,3.000000e+00,0.000000e+00,1.330975e+19,0.000000e+00,0.000000,4.360000e+04,1.0,1.0,0.0,1.0,1.0,NaN
max,8.865629e+06,6.945000e+03,8.000000e+02,2.301000e+04,1.839172e+19,9.020000e+03,1.000000,1.148299e+08,1.0,1.0,0.0,1.0,1.0,NaN


Find high missing columns: removes if > 95%

In [62]:
# Calculate missing data percentage for each column
missing_stats = pd.DataFrame({
    'column': valid_bookings.columns,
    'missing_count': valid_bookings.isnull().sum(),
    'missing_percentage': (valid_bookings.isnull().sum() / len(valid_bookings)) * 100
})

# Sort by missing percentage
missing_stats = missing_stats.sort_values('missing_percentage', ascending=False)

print("\nMissing Data Summary:")
print(missing_stats)

# Filter columns with less than 95% missing data
good_columns = missing_stats[missing_stats['missing_percentage'] < 99]['column'].tolist()

print(f"\n{len(good_columns)} columns have < 99% missing data")
print(f"{len(valid_bookings.columns) - len(good_columns)} columns will be dropped")

# Create filtered dataframe
valid_bookings_df = valid_bookings[good_columns]


Missing Data Summary:
                                  column  missing_count  missing_percentage
has_true_wallet          has_true_wallet        3267992          100.000000
guest_id_masked          guest_id_masked        3267512           99.985312
confirmed_by                confirmed_by        3266089           99.941769
has_wechat_pay            has_wechat_pay        3265440           99.921909
has_alipay                    has_alipay        3264530           99.894063
has_promptpay              has_promptpay        3172193           97.068567
cancelled_by                cancelled_by        3167337           96.919974
has_cc                            has_cc        3109817           95.159872
has_shopee_pay            has_shopee_pay        2588914           79.220329
has_special_request  has_special_request        2582512           79.024428
medium                            medium        2562347           78.407383
refund                            refund              1          

* Change denomination to Dollars
* Convert to booking date, start_time, end_time to datetime
* added day of week

In [63]:
#To convert current Revenue (denominated in cents) into dollars
valid_bookings_df['revenue_dollars'] = (valid_bookings_df['revenue']/100).round(2)
valid_bookings_df.drop(columns=['revenue'], inplace=True)

# Convert date and time columns to appropriate formats
valid_bookings_df['booking_date'] = pd.to_datetime(valid_bookings_df['date'], errors='coerce')
valid_bookings_df['start_time'] = pd.to_datetime(valid_bookings_df['start_time'], format="%H:%M", errors='coerce').dt.time
valid_bookings_df['end_time'] = pd.to_datetime(valid_bookings_df['end_time'], format="%H:%M", errors='coerce').dt.time
valid_bookings_df.drop(columns=['date'], inplace=True)
# Add weekday number (0=Monday, 6=Sunday)
valid_bookings_df['day_of_week_index'] = pd.to_datetime(valid_bookings_df['booking_date']).dt.weekday

# Add weekday name
valid_bookings_df['day_of_week'] = pd.to_datetime(valid_bookings_df['booking_date']).dt.day_name()

C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\2777764728.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df['revenue_dollars'] = (valid_bookings_df['revenue']/100).round(2)
C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\2777764728.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df.drop(columns=['revenue'], inplace=True)
C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\2777764728.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

In [64]:
#clear instance where a single user has multiple transactions recorded on the same date, with different revenue amounts. 
#clarified with Hungry Hub team that the duplicates exist because of the way they refresh and update their data, they mentioned can check a particular column (refer to google spreadsheet) but 
#they have yet to provide us with the column so for now we can't do much about it. 

valid_bookings_df.loc[valid_bookings_df['id'] == 8716589]

,has_promptpay,cancelled_by,has_cc,has_shopee_pay,has_special_request,medium,refund,end_time,start_time,id,is_temporary,for_locking_system,no_show,...,restaurant_id,created_at,ack,adjusted,active,prepared,revenue_dollars,booking_date,day_of_week_index,day_of_week
3182824,NaN,<NA>,NaN,0.0,1.0,Web,False,20:00:00,19:00:00,8716589,False,False,False,...,222,2025-12-16T05:34:03,True,False,True,False,999.0,2025-12-23,1,Tuesday
3182825,NaN,<NA>,NaN,0.0,1.0,Web,False,20:00:00,19:00:00,8716589,False,False,False,...,222,2025-12-16T05:34:03,True,False,True,False,1071.5,2025-12-23,1,Tuesday
3182826,NaN,<NA>,NaN,0.0,1.0,Web,False,20:00:00,19:00:00,8716589,False,False,False,...,222,2025-12-16T05:34:03,True,False,True,False,1098.9,2025-12-23,1,Tuesday


* removes if kids less than 0 or more than 100
* removes if adults less than 0 or more than 1000
* removes if revenue > 10000

In [65]:
# Define custom outlier conditions -- to determine our outliers based on Hungry Hub's requests/negative values/extreme values
valid_bookings_df['is_outlier'] = False
valid_bookings_df['outlier_reason'] = ''

# Kids outliers: < 0 or > 100
kids_outliers = (valid_bookings_df['kids'] < 0) | (valid_bookings_df['kids'] > 100)
valid_bookings_df.loc[kids_outliers, 'is_outlier'] = True
valid_bookings_df.loc[kids_outliers, 'outlier_reason'] += 'kids_invalid; '

# Adults outliers: < 0 or > 1000
adults_outliers = (valid_bookings_df['adult'] < 0) | (valid_bookings_df['adult'] > 1000)
valid_bookings_df.loc[adults_outliers, 'is_outlier'] = True
valid_bookings_df.loc[adults_outliers, 'outlier_reason'] += 'adults_invalid; '

# Revenue outliers: > 10000
revenue_outliers = valid_bookings_df['revenue_dollars'] > 10000
valid_bookings_df.loc[revenue_outliers, 'is_outlier'] = True
valid_bookings_df.loc[revenue_outliers, 'outlier_reason'] += 'revenue_high; '

# Summary
print("="*60)
print("OUTLIER DETECTION SUMMARY")
print("="*60)
print(f"Total records: {len(valid_bookings_df):,}")
print(f"Total outliers: {valid_bookings_df['is_outlier'].sum():,} ({valid_bookings_df['is_outlier'].sum()/len(valid_bookings_df)*100:.2f}%)")
print()

# Breakdown by type
print("Outlier breakdown:")
print(f"  Kids outliers (< 0 or > 100): {kids_outliers.sum():,}")
print(f"  Adults outliers (< 0 or > 1000): {adults_outliers.sum():,}")
print(f"  Revenue outliers (> $10000): {revenue_outliers.sum():,}")
print()

# Detailed statistics
outliers_df = valid_bookings_df[valid_bookings_df['is_outlier'] == True]

print("="*60)
print("KIDS OUTLIERS")
print("="*60)
kids_out = valid_bookings_df[kids_outliers]
if len(kids_out) > 0:
    print(f"Negative kids: {(kids_out['kids'] < 0).sum()}")
    print(f"Kids > 10: {(kids_out['kids'] > 10).sum()}")
    print(f"Max kids value: {kids_out['kids'].max()}")
    print(f"Min kids value: {kids_out['kids'].min()}")
    print("\nSample:")
    print(kids_out[['id', 'booking_date', 'kids', 'adult', 'revenue_dollars']].sort_values('kids', ascending=False).head(10))

print ()

OUTLIER DETECTION SUMMARY
Total records: 2,378,953
Total outliers: 59 (0.00%)

Outlier breakdown:
  Kids outliers (< 0 or > 100): 4
  Adults outliers (< 0 or > 1000): 0
  Revenue outliers (> $10000): 57

KIDS OUTLIERS
Negative kids: 1
Kids > 10: 3
Max kids value: 23010
Min kids value: -1

Sample:
              id booking_date   kids  adult  revenue_dollars
587850   4870618   2024-05-19  23010      1        1148298.8
2893858  7840206   2025-08-26  10000      9         647990.0
392856   4575452   2024-04-03    120      1           6087.8
3124387  8401289   2025-12-10     -1      2            233.2



C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\409366426.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df['is_outlier'] = False
C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\409366426.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df['outlier_reason'] = ''


In [66]:
print()
print("="*60)
print("ADULTS OUTLIERS")
print("="*60)
adults_out = valid_bookings_df[adults_outliers]
if len(adults_out) > 0:
    print(f"Negative adults: {(adults_out['adult'] < 0).sum()}")
    print(f"Adults > 50: {(adults_out['adult'] > 45).sum()}")
    print(f"Max adults value: {adults_out['adult'].max()}")
    print(f"Min adults value: {adults_out['adult'].min()}")
    print("\nSample:")
    print(adults_out[['id', 'booking_date', 'kids', 'adult', 'revenue_dollars']].sort_values('adult', ascending=True).head(10))


ADULTS OUTLIERS


In [67]:
print()
print("="*60)
print("REVENUE OUTLIERS")
print("="*60)
revenue_out = valid_bookings_df[revenue_outliers]
if len(revenue_out) > 0:
    print(f"Count: {len(revenue_out):,}")
    print(f"Max revenue: ${revenue_out['revenue_dollars'].max():,.2f}")
    print(f"Mean of outliers: ${revenue_out['revenue_dollars'].mean():,.2f}")
    print(f"Median of outliers: ${revenue_out['revenue_dollars'].median():,.2f}")
    print("\nTop 10 by revenue:")
    print(revenue_out[['id', 'booking_date', 'kids', 'adult', 'revenue_dollars']].sort_values('revenue_dollars', ascending=False).head(10))

# View all outliers sorted
print()
print("="*60)
print("ALL OUTLIERS (Top 10)")
print("="*60)
print(outliers_df[['id', 'booking_date', 'kids', 'adult', 'revenue_dollars', 'outlier_reason']].head(10))


REVENUE OUTLIERS
Count: 57
Max revenue: $1,148,298.80
Mean of outliers: $48,188.73
Median of outliers: $16,198.20

Top 10 by revenue:
              id booking_date   kids  adult  revenue_dollars
587850   4870618   2024-05-19  23010      1        1148298.8
2893858  7840206   2025-08-26  10000      9         647990.0
2757697  7564665   2025-06-14      0      2          41205.6
1963965  6587686   2024-12-22      0    200          39000.0
989111   5370143   2024-07-28      0     10          34500.0
554922   4806531   2024-07-03      0    300          29940.0
1963961  6587680   2024-12-22      0    150          29250.0
2681678  7450727   2025-05-24      0    300          28620.0
26474    4084513   2024-03-02      0      2          26997.0
26473    4084512   2024-03-01      0      2          26997.0

ALL OUTLIERS (Top 10)
             id booking_date  kids  adult  revenue_dollars  outlier_reason
7749    4039726   2024-01-19     0     46          10110.8  revenue_high; 
26473   4084512   202

In [68]:
valid_bookings_df.head()

,has_promptpay,cancelled_by,has_cc,has_shopee_pay,has_special_request,medium,refund,end_time,start_time,id,is_temporary,for_locking_system,no_show,...,ack,adjusted,active,prepared,revenue_dollars,booking_date,day_of_week_index,day_of_week,is_outlier,outlier_reason
0,NaN,<NA>,NaN,0.0,0.0,iOS,<NA>,17:00:00,15:00:00,865732,False,False,False,...,True,False,True,True,553.0,2025-12-01,0,Monday,False,
1,NaN,admin,NaN,0.0,0.0,Web,False,18:00:00,16:00:00,2137851,False,False,False,...,True,False,False,False,79.20,2024-10-31,3,Thursday,False,
2,NaN,admin,NaN,0.0,1.0,Web,False,20:00:00,18:00:00,2137857,False,False,False,...,True,False,False,False,79.20,2024-10-31,3,Thursday,False,
5,NaN,<NA>,NaN,0.0,1.0,Vendor_App,False,19:00:00,19:00:00,3391224,False,False,False,...,True,False,True,False,149.93,2025-05-13,1,Tuesday,False,
6,NaN,<NA>,NaN,0.0,1.0,Vendor_App,False,19:00:00,19:00:00,3440352,False,False,False,...,True,False,True,False,302.25,2024-06-17,0,Monday,False,


In [69]:
valid_bookings_df.to_parquet("valid_bookings.parquet")

* aggregate restaurant level

In [70]:
#Start of restaurant-level aggregation, feature engineering right now currently includes just the aggregation of numeric features. 
#One possible enhancement is to include categorical features

from scipy import stats
from sklearn.preprocessing import StandardScaler

# Create total_guests if not exists
if 'total_guests' not in valid_bookings_df.columns:
    valid_bookings_df['total_guests'] = valid_bookings_df['adult'] + valid_bookings_df['kids']

# Create revenue_per_guest if not exists
if 'revenue_per_guest' not in valid_bookings_df.columns:
    valid_bookings_df['revenue_per_guest'] = (valid_bookings_df['revenue_dollars'] / valid_bookings_df['total_guests']).replace([np.inf, -np.inf], 0).fillna(0).round(2)

# restaurant-level aggregation
restaurant_agg = valid_bookings_df.groupby('restaurant_id').agg({
    'id': 'count',
    'revenue_dollars': ['sum', 'mean', 'median', 'std', 'min', 'max'],
    'adult': ['sum', 'mean', 'median'],
    'kids': ['sum', 'mean', 'median'],
    'total_guests': ['sum', 'mean', 'median', 'std'],
    'revenue_per_guest': ['mean', 'median', 'std'],
    'booking_date': ['min', 'max', 'nunique'],
    'day_of_week': lambda x: x.mode()[0] if len(x.mode()) > 0 else None,  # Most common day
    'start_time': ['min', 'max']
}).round(2)

# Flatten column names
restaurant_agg.columns = ['_'.join(col).strip() for col in restaurant_agg.columns.values]
restaurant_agg = restaurant_agg.reset_index()

C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\1422537291.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df['total_guests'] = valid_bookings_df['adult'] + valid_bookings_df['kids']
C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\1422537291.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df['revenue_per_guest'] = (valid_bookings_df['revenue_dollars'] / valid_bookings_df['total_guests']).replace([np.inf, -np.inf], 0).fillna(0).round(2)


In [71]:
import numpy as np
import pandas as pd

# -----------------------------
# 0) Ensure categorical columns exist + are clean
# -----------------------------
categorical_cols = [
    "medium",
    "arrived",
    "has_special_request",
    "has_promptpay",
    "has_cc",
    "has_shopee_pay",
    "has_alipay",
    "has_wechat_pay",
    "has_true_wallet",
]

# Keep only cols that actually exist (prevents KeyError)
categorical_cols = [c for c in categorical_cols if c in valid_bookings_df.columns]

# Optional: standardize booleans to True/False (helps OHE consistency)
bool_like = [c for c in categorical_cols if c != "medium"]
for c in bool_like:
    valid_bookings_df[c] = valid_bookings_df[c].astype("bool", errors="ignore")

# Optional: fill missing medium as "unknown" so it gets its own bucket
if "medium" in categorical_cols:
    valid_bookings_df["medium"] = valid_bookings_df["medium"].fillna("unknown").astype(str)

# -----------------------------
# 1) One-hot encode booking-level categorical vars
# -----------------------------
group_var = "restaurant_id"

bookings_ohe = pd.get_dummies(
    valid_bookings_df[[group_var] + categorical_cols].copy(),
    columns=categorical_cols,
    prefix=categorical_cols,
    prefix_sep="__",
    dummy_na=False,
)

# Grab ONLY the dummy columns we created
dummy_cols = [c for c in bookings_ohe.columns if c != group_var]

# -----------------------------
# 2) Aggregate dummies per restaurant: mean + sum
#    - mean => "share of bookings with that attribute"
#    - sum  => "count of bookings with that attribute"
# -----------------------------
cat_grouped = bookings_ohe.groupby(group_var)[dummy_cols].agg(["mean", "sum"])

# Flatten columns into <dummy>_count_norm and <dummy>_count
cat_grouped.columns = [
    f"{col}_count_norm" if stat == "mean" else f"{col}_count"
    for col, stat in cat_grouped.columns
]
cat_grouped = cat_grouped.reset_index()

# -----------------------------
# 3) Merge into your restaurant_agg
# -----------------------------
restaurant_agg = restaurant_agg.merge(
    cat_grouped,
    on="restaurant_id",
    how="left",
)

# If a restaurant somehow has no rows in cat_grouped, fill 0 for those dummy aggregations
new_cat_cols = [c for c in restaurant_agg.columns if c.endswith("_count") or c.endswith("_count_norm")]
restaurant_agg[new_cat_cols] = restaurant_agg[new_cat_cols].fillna(0)

restaurant_agg.head()


C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\2485522981.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df[c] = valid_bookings_df[c].astype("bool", errors="ignore")
C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\2485522981.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_bookings_df[c] = valid_bookings_df[c].astype("bool", errors="ignore")
C:\Users\lowth\AppData\Local\Temp\ipykernel_11972\2485522981.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy o

,restaurant_id,id_count,revenue_dollars_sum,revenue_dollars_mean,revenue_dollars_median,revenue_dollars_std,revenue_dollars_min,revenue_dollars_max,adult_sum,adult_mean,...,arrived__True_count_norm,arrived__True_count,has_special_request__False_count_norm,has_special_request__False_count,has_special_request__True_count_norm,has_special_request__True_count,has_shopee_pay__False_count_norm,has_shopee_pay__False_count,has_shopee_pay__True_count_norm,has_shopee_pay__True_count
0,33,8546,1909944.62,223.49,114.4,268.61,7.20,7052.0,30346,3.55,...,0.220337,1883,0.241516,2064,0.758484,6482,0.279195,2386,0.720805,6160
1,34,196,52627.80,268.51,229.0,182.07,169.00,1436.0,530,2.70,...,0.178571,35,0.290816,57,0.709184,139,0.428571,84,0.571429,112
2,168,314,83182.80,264.91,199.8,189.98,99.00,1298.7,798,2.54,...,0.085987,27,0.414013,130,0.585987,184,0.585987,184,0.414013,130
3,220,326,185774.20,569.86,690.9,380.52,192.60,2072.7,1128,3.46,...,0.088957,29,0.325153,106,0.674847,220,0.463190,151,0.536810,175
4,222,10560,5459723.62,517.02,370.8,483.19,79.68,12236.4,48400,4.58,...,0.213068,2250,0.254545,2688,0.745455,7872,0.306534,3237,0.693466,7323


In [20]:
restaurant_agg.columns
# Rename for clarity
rename_dict = {
    'id_count': 'total_bookings',
    'revenue_dollars_sum': 'total_revenue',
    'revenue_dollars_mean': 'avg_revenue_per_booking',
    'revenue_dollars_median': 'median_revenue_per_booking',
    'revenue_dollars_std': 'revenue_std',
    'revenue_dollars_min': 'min_revenue',
    'revenue_dollars_max': 'max_revenue',
    'adult_sum': 'total_adults',
    'adult_mean': 'avg_adults',
    'adult_median': 'median_adults',
    'kids_sum': 'total_kids',
    'kids_mean': 'avg_kids',
    'kids_median': 'median_kids',
    'total_guests_sum': 'total_guests_sum',
    'total_guests_mean': 'avg_party_size',
    'total_guests_median': 'median_party_size',
    'total_guests_std': 'party_size_std',
    'revenue_per_guest_mean': 'avg_revenue_per_guest',
    'revenue_per_guest_median': 'median_revenue_per_guest',
    'revenue_per_guest_std': 'revenue_per_guest_std',
    'booking_date_min': 'first_booking',
    'booking_date_max': 'last_booking',
    'booking_date_nunique': 'unique_booking_days',
    'day_of_week_<lambda>': 'most_popular_day_of_week',
    'start_time_min': 'earliest_booking_time',
    'start_time_max': 'latest_booking_time'
}

restaurant_agg.rename(columns=rename_dict, inplace=True)

restaurant_agg.head()

,restaurant_id,total_bookings,total_revenue,avg_revenue_per_booking,median_revenue_per_booking,revenue_std,min_revenue,max_revenue,total_adults,avg_adults,...,arrived__True_count_norm,arrived__True_count,has_special_request__False_count_norm,has_special_request__False_count,has_special_request__True_count_norm,has_special_request__True_count,has_shopee_pay__False_count_norm,has_shopee_pay__False_count,has_shopee_pay__True_count_norm,has_shopee_pay__True_count
0,33,8546,1909944.62,223.49,114.4,268.61,7.20,7052.0,30346,3.55,...,0.220337,1883,0.241516,2064,0.758484,6482,0.279195,2386,0.720805,6160
1,34,196,52627.80,268.51,229.0,182.07,169.00,1436.0,530,2.70,...,0.178571,35,0.290816,57,0.709184,139,0.428571,84,0.571429,112
2,168,314,83182.80,264.91,199.8,189.98,99.00,1298.7,798,2.54,...,0.085987,27,0.414013,130,0.585987,184,0.585987,184,0.414013,130
3,220,326,185774.20,569.86,690.9,380.52,192.60,2072.7,1128,3.46,...,0.088957,29,0.325153,106,0.674847,220,0.463190,151,0.536810,175
4,222,10560,5459723.62,517.02,370.8,483.19,79.68,12236.4,48400,4.58,...,0.213068,2250,0.254545,2688,0.745455,7872,0.306534,3237,0.693466,7323


In [74]:
restaurants_google_df = pd.read_csv('places_api_new_results.csv')

restaurants_google_df.head()

,input_string,found,official_name,city,country,formatted_address,rating,website,error
0,Audrey Cafe Thonglor Soi 11,True,AUDREY CAFE & BISTRO,NaN,Thailand,"Soi Thong Lo 11, Khwaeng Khlong Tan Nuea, Watt...",4.2,http://www.audreygroup.com/,NaN
1,Attico Cucina Italiana Radisson Blu Plaza Bangkok,True,Attico Cucina Italiana at Radisson Blu Plaza B...,NaN,Thailand,"489 Sukhumvit Rd, Khwaeng Khlong Toei Nuea, Wa...",4.5,https://www.radissonhotels.com/en-us/hotels/ra...,NaN
2,The Living Room at Sheraton Grande Sukhumvit A...,True,"Sheraton Grande Sukhumvit, a Luxury Collection...",NaN,Thailand,"250 Sukhumvit Rd, Khwaeng Khlong Toei, Khet Kh...",4.7,https://www.marriott.com/en-us/hotels/bkklc-sh...,NaN
3,Rain Tree Cafe at The Athenee Hotel,True,Rain Tree Café,NaN,Thailand,"61 Thanon Witthayu, Khwaeng Lumphini, Pathum W...",4.4,https://www.raintreecafebangkok.com/,NaN
4,Skyline at Avani+ Riverside Bangkok Hotel,True,Skyline,NaN,Thailand,"F11, AVANI RIVERSIDE HOTEL, 257 ถ. เจริญนคร Sa...",4.2,https://www.avanihotels.com/en/riverside-bangk...,NaN


In [27]:
restaurants_google_df['country'].value_counts()

country
Thailand           1916
Singapore           325
Malaysia             31
United States        28
Japan                 3
Indonesia             3
India                 3
Vietnam               2
Australia             2
United Kingdom        2
Netherlands           2
Nigeria               1
Brazil                1
Portugal              1
France                1
China                 1
North Macedonia       1
Canada                1
Greece                1
Hong Kong             1
Italy                 1
New Zealand           1
Pakistan              1
Name: count, dtype: int64